# 02. Factor Research Workbench
**Objective:** Prototype, validate, and analyze the decay of new alpha factors.

### Workflow:
1. Define Factor Logic (pandas/numpy).
2. **IC Analysis:** Information Coefficient over time.
3. **Quantile Analysis:** Monotonicity of returns.
4. **Alpha Decay:** How fast does the signal fade?

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('../').resolve()
sys.path.append(str(PROJECT_ROOT))

from config.settings import config
from quant_alpha.utils import load_parquet
from quant_alpha.research.factor_analysis import FactorAnalyzer
from quant_alpha.research.alpha_decay import AlphaDecayAnalyzer

%matplotlib inline

## 1. Load Data & Define Factor
Example: **Volatility-Adjusted Momentum** (Return / StdDev)

In [ ]:
df = load_parquet(config.CACHE_DIR / 'master_data_with_factors.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'])

# --- PROTOTYPE FACTOR HERE ---
window = 20
df['mom_20'] = df.groupby('ticker')['close'].pct_change(window)
df['vol_20'] = df.groupby('ticker')['close'].transform(lambda x: x.pct_change().rolling(window).std())

df['alpha_signal'] = df['mom_20'] / (df['vol_20'] + 1e-8)

# Clean up
analysis_df = df.dropna(subset=['alpha_signal', 'raw_ret_5d']).copy()
print(f"Factor defined. Analysis set: {len(analysis_df):,} rows.")

## 2. Information Coefficient (IC) Analysis

In [ ]:
analyzer = FactorAnalyzer(analysis_df, factor_col='alpha_signal', forward_ret_col='raw_ret_5d')

print("--- IC Summary ---")
print(analyzer.get_ic_summary())

analyzer.plot_ic_ts(window=20)

## 3. Quantile Analysis
Does the top quintile consistently outperform the bottom?

In [ ]:
analyzer.plot_quantile_returns(quantiles=5)

## 4. Alpha Decay Analysis
How long does the signal remain predictive?

In [ ]:
decay = AlphaDecayAnalyzer(analysis_df, factor_col='alpha_signal')
decay.plot_decay(max_horizon=10)